# Recalculate Few-Shot Summary Statistics

This notebook recalculates the summary statistics from the individual results CSV file,
including F1 Weighted which was missing from the original summary.

**Methodology:** Trimmed mean (remove max and min, average remaining 8 scores)

In [7]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

In [8]:
# Load individual results
df = pd.read_csv('few_shot_individual_results_indomalay.csv')
print(f"Loaded {len(df)} individual experiment results")
print(f"Shot sizes: {sorted(df['n_shot'].unique())}")
print(f"Seeds per shot: {df.groupby('n_shot').size().values[0]}")
df.head()

Loaded 50 individual experiment results
Shot sizes: [np.int64(2), np.int64(4), np.int64(8), np.int64(16), np.int64(100)]
Seeds per shot: 10


,n_shot,seed,train_size,val_size,test_size,accuracy,precision_binary,recall_binary,f1_binary,roc_auc,precision_macro,recall_macro,f1_macro,precision_weighted,recall_weighted,f1_weighted,best_val_f1
0,2,42,4,4,25033,0.654416,0.657631,0.990836,0.790558,0.579358,0.462311,0.498633,0.401557,0.524130,0.654416,0.524677,0.666667
1,2,123,4,4,25033,0.435106,0.675688,0.272727,0.388603,0.497312,0.511873,0.510297,0.431819,0.563721,0.435106,0.418141,0.666667
2,2,456,4,4,25033,0.399153,0.734268,0.136667,0.230443,0.523131,0.543315,0.520701,0.368817,0.603752,0.399153,0.325021,0.666667
3,2,789,4,4,25033,0.545400,0.718087,0.509346,0.595967,0.595845,0.556121,0.562095,0.538166,0.607384,0.545400,0.556460,0.000000
4,2,1024,4,4,25033,0.575241,0.662190,0.724117,0.691770,0.524249,0.507029,0.506302,0.504405,0.556138,0.575241,0.563707,0.800000


In [9]:
def compute_trimmed_mean(scores):
    """Remove max and min, average the rest (trimmed mean)"""
    if len(scores) <= 2:
        return np.mean(scores)
    scores_sorted = sorted(scores)
    trimmed_scores = scores_sorted[1:-1]  # Remove first (min) and last (max)
    return np.mean(trimmed_scores)

def compute_trimmed_std(scores):
    """Compute std of trimmed scores"""
    if len(scores) <= 2:
        return np.std(scores)
    scores_sorted = sorted(scores)
    trimmed_scores = scores_sorted[1:-1]
    return np.std(trimmed_scores)

In [10]:
# Metrics to aggregate
metrics = ['accuracy', 'precision_binary', 'recall_binary', 'f1_binary', 
           'roc_auc', 'precision_macro', 'recall_macro', 'f1_macro',
           'precision_weighted', 'recall_weighted', 'f1_weighted']

# Aggregate results by shot size
aggregated_results = []

for n_shot in sorted(df['n_shot'].unique()):
    shot_df = df[df['n_shot'] == n_shot]
    
    agg = {'n_shot': n_shot, 'num_runs': len(shot_df)}
    
    for metric in metrics:
        scores = shot_df[metric].tolist()
        agg[f'{metric}_mean'] = compute_trimmed_mean(scores)
        agg[f'{metric}_std'] = compute_trimmed_std(scores)
        agg[f'{metric}_all'] = scores
    
    aggregated_results.append(agg)
    
print(f"Aggregated {len(aggregated_results)} shot sizes")

Aggregated 5 shot sizes


In [11]:
# Create summary dataframe with F1 Weighted included
summary_data = []
for agg in aggregated_results:
    summary_data.append({
        'n_shot': agg['n_shot'],
        'num_runs': agg['num_runs'],
        'accuracy': f"{agg['accuracy_mean']:.4f} ± {agg['accuracy_std']:.4f}",
        'precision': f"{agg['precision_binary_mean']:.4f} ± {agg['precision_binary_std']:.4f}",
        'recall': f"{agg['recall_binary_mean']:.4f} ± {agg['recall_binary_std']:.4f}",
        'f1_binary': f"{agg['f1_binary_mean']:.4f} ± {agg['f1_binary_std']:.4f}",
        'roc_auc': f"{agg['roc_auc_mean']:.4f} ± {agg['roc_auc_std']:.4f}",
        'f1_macro': f"{agg['f1_macro_mean']:.4f} ± {agg['f1_macro_std']:.4f}",
        'f1_weighted': f"{agg['f1_weighted_mean']:.4f} ± {agg['f1_weighted_std']:.4f}",
    })

summary_df = pd.DataFrame(summary_data)
summary_df.to_csv('few_shot_summary_indomalay.csv', index=False)
print("Saved: few_shot_summary_indomalay.csv")
summary_df

Saved: few_shot_summary_indomalay.csv


,n_shot,num_runs,accuracy,precision,recall,f1_binary,roc_auc,f1_macro,f1_weighted
0,2,10,0.5345 ± 0.0723,0.6809 ± 0.0285,0.5858 ± 0.2259,0.6007 ± 0.1270,0.5393 ± 0.0339,0.4574 ± 0.0438,0.5096 ± 0.0506
1,4,10,0.5191 ± 0.0965,0.6787 ± 0.0320,0.5362 ± 0.2836,0.5514 ± 0.1882,0.5341 ± 0.0480,0.4427 ± 0.0554,0.4783 ± 0.0836
2,8,10,0.5543 ± 0.0758,0.6857 ± 0.0322,0.6108 ± 0.2268,0.6228 ± 0.1175,0.5558 ± 0.0489,0.4811 ± 0.0444,0.5271 ± 0.0427
3,16,10,0.6047 ± 0.0527,0.6959 ± 0.0266,0.7087 ± 0.1079,0.7001 ± 0.0576,0.5791 ± 0.0521,0.5335 ± 0.0595,0.5870 ± 0.0472
4,100,10,0.6895 ± 0.0125,0.7513 ± 0.0136,0.7970 ± 0.0333,0.7721 ± 0.0099,0.6927 ± 0.0255,0.6424 ± 0.0158,0.6835 ± 0.0128


In [12]:
# Save detailed JSON results
json_results = {
    'experiment_info': {
        'model': 'indolem/indobert-base-uncased',
        'dataset': 'Indo-Malay',
        'shot_sizes': [int(x) for x in sorted(df['n_shot'].unique().tolist())],
        'num_seeds': 10,
        'methodology': 'Trimmed mean (remove max and min, average remaining 8)',
        'recalculated_at': datetime.now().isoformat()
    },
    'aggregated_results': []
}

for agg in aggregated_results:
    result_entry = {
        'n_shot': int(agg['n_shot']),
        'num_runs': int(agg['num_runs']),
        'metrics': {}
    }
    for metric in metrics:
        result_entry['metrics'][metric] = {
            'trimmed_mean': float(agg[f'{metric}_mean']),
            'trimmed_std': float(agg[f'{metric}_std']),
            'all_scores': [float(x) for x in agg[f'{metric}_all']]
        }
    json_results['aggregated_results'].append(result_entry)

with open('few_shot_results_indomalay.json', 'w') as f:
    json.dump(json_results, f, indent=2)
print("Saved: few_shot_results_indomalay.json")

Saved: few_shot_results_indomalay.json


In [13]:
# Save human-readable text summary
with open('few_shot_summary_indomalay.txt', 'w') as f:
    f.write("KAPALM Few-Shot Learning Results - Indo-Malay Dataset\n")
    f.write("="*120 + "\n\n")
    f.write("Experimental Setup:\n")
    f.write(f"  - Model: indolem/indobert-base-uncased (IndoBERT)\n")
    f.write(f"  - Shot sizes: {sorted(df['n_shot'].unique().tolist())}\n")
    f.write(f"  - Number of random seeds: 10\n")
    f.write(f"  - Scoring: Trimmed mean (remove best and worst, average remaining 8)\n\n")
    f.write("="*120 + "\n\n")
    f.write(f"{'K-Shot':<10} {'Accuracy':<20} {'F1 (Binary)':<20} {'ROC-AUC':<20} {'F1 (Macro)':<20} {'F1 (Weighted)':<20}\n")
    f.write("-"*120 + "\n")
    
    for agg in aggregated_results:
        f.write(f"{agg['n_shot']:<10} "
               f"{agg['accuracy_mean']:.4f} ± {agg['accuracy_std']:.4f}    "
               f"{agg['f1_binary_mean']:.4f} ± {agg['f1_binary_std']:.4f}    "
               f"{agg['roc_auc_mean']:.4f} ± {agg['roc_auc_std']:.4f}    "
               f"{agg['f1_macro_mean']:.4f} ± {agg['f1_macro_std']:.4f}    "
               f"{agg['f1_weighted_mean']:.4f} ± {agg['f1_weighted_std']:.4f}\n")
    
    f.write("="*120 + "\n")

print("Saved: few_shot_summary_indomalay.txt")

Saved: few_shot_summary_indomalay.txt


In [14]:
# Print final results table
print("\n" + "="*120)
print("FINAL RESULTS TABLE (with F1 Weighted)")
print("="*120)
print(f"{'K-Shot':<10} {'Accuracy':<20} {'F1 (Binary)':<20} {'ROC-AUC':<20} {'F1 (Macro)':<20} {'F1 (Weighted)':<20}")
print("-"*120)

for agg in aggregated_results:
    print(f"{agg['n_shot']:<10} "
          f"{agg['accuracy_mean']:.4f} ± {agg['accuracy_std']:.4f}    "
          f"{agg['f1_binary_mean']:.4f} ± {agg['f1_binary_std']:.4f}    "
          f"{agg['roc_auc_mean']:.4f} ± {agg['roc_auc_std']:.4f}    "
          f"{agg['f1_macro_mean']:.4f} ± {agg['f1_macro_std']:.4f}    "
          f"{agg['f1_weighted_mean']:.4f} ± {agg['f1_weighted_std']:.4f}")

print("="*120)
print("\n✅ All summary files updated with F1 Weighted!")


FINAL RESULTS TABLE (with F1 Weighted)
K-Shot     Accuracy             F1 (Binary)          ROC-AUC              F1 (Macro)           F1 (Weighted)       
------------------------------------------------------------------------------------------------------------------------
2          0.5345 ± 0.0723    0.6007 ± 0.1270    0.5393 ± 0.0339    0.4574 ± 0.0438    0.5096 ± 0.0506
4          0.5191 ± 0.0965    0.5514 ± 0.1882    0.5341 ± 0.0480    0.4427 ± 0.0554    0.4783 ± 0.0836
8          0.5543 ± 0.0758    0.6228 ± 0.1175    0.5558 ± 0.0489    0.4811 ± 0.0444    0.5271 ± 0.0427
16         0.6047 ± 0.0527    0.7001 ± 0.0576    0.5791 ± 0.0521    0.5335 ± 0.0595    0.5870 ± 0.0472
100        0.6895 ± 0.0125    0.7721 ± 0.0099    0.6927 ± 0.0255    0.6424 ± 0.0158    0.6835 ± 0.0128

✅ All summary files updated with F1 Weighted!
